In [ ]:
!pip install gradio pandas openpyxl requests -q

import json
import pandas as pd
import gradio as gr
import requests
import re
import difflib

# ----------------------------------------------------
# 0. LLM CONFIG
# ----------------------------------------------------
HF_TOKEN = "<use key from attached file>"   # <<< YOUR HF TOKEN
HF_URL = "https://router.huggingface.co/v1/chat/completions"

HF_MODEL = "Qwen/Qwen2.5-72B-Instruct"  # or another strong instruct model

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json",
}

def call_llm(system_prompt, user_prompt, max_tokens=512):
    data = {
        "model": HF_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
    }
    resp = requests.post(HF_URL, headers=headers, json=data)
    if resp.status_code != 200:
        return f"[LLM ERROR {resp.status_code}] {resp.text}"
    return resp.json()["choices"][0]["message"]["content"]


# ----------------------------------------------------
# 1. GLOBAL DATA
# ----------------------------------------------------
global_df = None
course_vocab = set()

EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/content/extra_50_universities_real_names.xlsx"


# ----------------------------------------------------
# 2. LOAD EXCEL AND BUILD VOCAB
# ----------------------------------------------------
def load_excel(path: str):
    global global_df, course_vocab
    try:
        df = pd.read_excel(path)
    except Exception as e:
        print(f"Could not read the Excel file at {path}: {e}")
        global_df = None
        course_vocab = set()
        return False

    required = [
        "University Name",
        "Tech Type Normalized",
        "Delivery Method Normalized",
        "Financial Cost",
        "Rating",
        "Courses List",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"Missing required columns: {missing}")
        global_df = None
        course_vocab = set()
        return False

    def ensure_list(x):
        if isinstance(x, list):
            return x
        if isinstance(x, str):
            try:
                return json.loads(x.replace("'", '"'))
            except Exception:
                return [p.strip() for p in x.split(",")]
        return []

    df["Courses List"] = df["Courses List"].apply(ensure_list)
    global_df = df

    names = []
    for courses in df["Courses List"]:
        for c in courses:
            names.append(str(c).strip())
    course_vocab = {c for c in names if c}

    print("Excel file loaded successfully from fixed path.")
    return True


DATA_LOADED = load_excel(EXCEL_PATH)


# ----------------------------------------------------
# 3. SUBJECT NORMALIZATION AND FILTERING
# ----------------------------------------------------
def extract_subject_from_text(text: str) -> str:
    t = text.lower()

    patterns = [
        r"study\s+([a-zA-Z ]+)",
        r"learn\s+([a-zA-Z ]+)",
        r"do\s+([a-zA-Z ]+)",
        r"interested in\s+([a-zA-Z ]+)",
        r"want to do\s+([a-zA-Z ]+)",
        r"want to study\s+([a-zA-Z ]+)",
        r"want to learn\s+([a-zA-Z ]+)",
    ]
    for pat in patterns:
        m = re.search(pat, t)
        if m:
            candidate = m.group(1).strip()
            if candidate:
                return candidate.title()

    words = re.findall(r"[a-zA-Z]+", t)
    stop = {"i", "want", "to", "do", "study", "learn", "about", "the", "a", "an"}
    content_words = [w for w in words if w not in stop]
    if not content_words:
        return text.title()
    if len(content_words) >= 2:
        subject = " ".join(content_words[-2:])
    else:
        subject = content_words[-1]
    return subject.title()

def normalize_interest_fuzzy(interest: str) -> str:
    if not course_vocab:
        return interest.title()
    candidates = list(course_vocab)
    match = difflib.get_close_matches(interest, candidates, n=1, cutoff=0.4)
    if match:
        return match[0]
    return interest.title()

def map_interest_to_keywords(interest: str):
    i = interest.lower()

    if "engineer" in i:
        return ["Engineering", "Mechanical Engineering", "Mechanical"]
    if "business" in i:
        return ["Business"]
    if "computer" in i or "cs" in i or "software" in i:
        return ["Computer Science", "Information Technology", "IT"]

    normalized = normalize_interest_fuzzy(interest)
    return [normalized]

def base_filter_for_interest_and_delivery(interest, delivery):
    if global_df is None:
        return None

    df = global_df.copy()
    keywords = map_interest_to_keywords(interest)

    delivery_norm = (
        df["Delivery Method Normalized"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace("-", " ", regex=False)
    )

    d = delivery.lower()
    if d in ["in-person", "in person"]:
        mask = delivery_norm.str.contains("in person") | delivery_norm.str.contains("on campus")
        df = df[mask]
    elif d == "online":
        mask = (
            delivery_norm.str.contains("online")
            | delivery_norm.str.contains("remote")
            | delivery_norm.str.contains("distance")
        )
        df = df[mask]
    # "both" means no delivery filter

    def matches_courses(row):
        row_courses = [str(c).lower() for c in row["Courses List"]]
        for k in keywords:
            k_low = k.lower()
            for c in row_courses:
                if k_low in c or c in k_low:
                    return True
        return False

    df = df[df.apply(matches_courses, axis=1)]
    return df

def any_for_interest_and_delivery(interest, delivery):
    df = base_filter_for_interest_and_delivery(interest, delivery)
    if df is None:
        return False
    return not df.empty

def filter_and_rank(interest, delivery, budget):
    df = base_filter_for_interest_and_delivery(interest, delivery)
    if df is None or df.empty:
        return []

    if budget:
        df = df[df["Financial Cost"] <= budget]
    if df.empty:
        return []

    def score(row):
        r = float(row["Rating"])
        b_bonus = (budget - row["Financial Cost"]) / max(budget, 1)
        return r * 2 + b_bonus

    df["Score"] = df.apply(score, axis=1)
    df = df.sort_values("Score", ascending=False)

    results = []
    for _, row in df.head(3).iterrows():
        results.append(
            f"**{row['University Name']}**\n"
            f"- Total Price: ${row['Financial Cost']}\n"
            f"- Rating: {row['Rating']}\n"
            f"- Courses: {', '.join(row['Courses List'])}\n"
        )
    return results


# ----------------------------------------------------
# 4. LLM PROMPTS
# ----------------------------------------------------
SYSTEM_PROMPT = """
You are UniGuide, a friendly university recommendation assistant.
Be warm, polite and concise.
Ask one simple question at a time.
Do not invent universities that are not in the dataset.
Always show total price clearly when recommending.
Respond naturally to what the user writes, even if it is a single word.
Only use the universities that are passed to you in the prompt when listing options.
"""

def llm_prompt(step, state, msg=None, recs=None):
    recs_text = "\n".join(recs or [])
    msg_text = msg if msg is not None else ""

    if step == 0:
        return """
Greet the user as UniGuide.
Briefly explain that you can recommend universities based on their subject interest, delivery preference and budget.
Then ask: "What subject are you interested in studying?"
"""

    if step == 1:
        return f"""
The user wrote about their interest: "{msg_text}".
After cleaning the text we interpreted the subject as: "{state['interest']}".

Reply in a friendly tone:
1) Acknowledge their interest and encourage them.
2) Then clearly ask: "Do you prefer in-person, online, or both?"
"""

    if step == 2:
        return f"""
The user expressed a delivery preference with: "{msg_text}".
We interpreted and stored the delivery mode as: "{state['delivery']}".

Reply in a friendly tone:
1) Confirm their delivery preference.
2) Then ask: "What is your total budget? (number only, for example 30000)"
"""

    if step == 3 and recs:
        return f"""
The user answered about budget with: "{msg_text}".
Current subject of interest: "{state['interest']}".
Chosen delivery mode: "{state['delivery']}".

Here are the matching universities from the dataset (do not add any others):

{recs_text}

Summarize these matches in a friendly way and highlight why they are good fits.
Then ask: "Would you like to explore anything else?"
"""

    if step == 4:
        return f"""
The user replied after seeing recommendations: "{msg_text}".

If their message clearly means they are done, respond politely and ask:
"How would you rate this experience from 1 to 5?"

If they seem to want more help, respond warmly and ask:
"What subject would you like to explore next?"
"""

    if step == 5:
        return f"""
The user replied with: "{msg_text}" when asked to rate the experience from 1 to 5.

If it looks like a valid rating between 1 and 5, thank them and ask:
"Would you like to explore another subject? (yes/no)"

If it does not look like a number between 1 and 5, politely say:
"Please rate from 1 to 5."
"""

    if step == 6:
        return f"""
The user replied: "{msg_text}" when asked if they would like to explore another subject.

If it clearly means yes, respond cheerfully and ask:
"Great, what subject would you like to explore next?"

If it clearly means no, respond politely and end the conversation, for example:
"Thanks for using UniGuide. Have a wonderful day."

If it is unclear, ask them to reply yes or no.
"""

    return f"""
The user said: "{msg_text}".
End the conversation politely and thank the user.
"""


# ----------------------------------------------------
# 5. HELPER TO INTERPRET DELIVERY
# ----------------------------------------------------
def interpret_delivery(msg: str):
    t = msg.lower()

    if re.search(r"\b1\b", t) or "option 1" in t:
        return "in-person"
    if re.search(r"\b2\b", t) or "option 2" in t:
        return "online"
    if re.search(r"\b3\b", t) or "option 3" in t:
        return "both"

    if "online" in t and "in person" not in t and "on campus" not in t and "campus" not in t:
        return "online"
    if "in person" in t or "on campus" in t or "campus" in t or "classroom" in t:
        return "in-person"
    if "both" in t or "either" in t or "no preference" in t or "hybrid" in t:
        return "both"

    t_stripped = t.strip()
    if t_stripped == "online":
        return "online"
    if t_stripped in ["in-person", "in person"]:
        return "in-person"
    if t_stripped == "both":
        return "both"

    return None


# ----------------------------------------------------
# 6. CONVERSATION ENGINE
# ----------------------------------------------------
def step_logic(msg, state):
    if global_df is None:
        return (
            "I am having trouble loading the university dataset. "
            "Please verify the Excel path and reload the app.",
            state,
        )

    step = state["step"]

    if step == 0:
        state["step"] = 1
        return call_llm(SYSTEM_PROMPT, llm_prompt(0, state)), state

    if step == 1:
        state["interest_raw"] = msg
        cleaned = extract_subject_from_text(msg)
        state["interest"] = cleaned
        state["step"] = 2
        return call_llm(SYSTEM_PROMPT, llm_prompt(1, state, msg=msg)), state

    if step == 2:
        delivery = interpret_delivery(msg)
        if delivery is None:
            prompt = f"""
The user replied: "{msg}" when asked about delivery method.

Politely explain that you need to know if they prefer:
- in-person
- online
- both

Tell them they can either type the full word, or reply with:
1 for in-person
2 for online
3 for both

Ask them again for their preferred delivery mode.
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

        state["delivery"] = delivery

        if not any_for_interest_and_delivery(state["interest"], state["delivery"]):
            prompt = f"""
The user chose a delivery option that we interpret as "{state['delivery']}" for the subject "{state['interest']}",
but there are no universities with that delivery mode in the dataset.

Politely explain this and ask them to choose a different delivery option.
Remind them of the choices:
1. In-person
2. Online
3. Both (online and in-person)

Tell them they can answer with the word or the number.
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

        state["step"] = 3
        return call_llm(SYSTEM_PROMPT, llm_prompt(2, state, msg=msg)), state

    if step == 3:
        nums = re.findall(r"\d+", msg)
        if not nums:
            prompt = f"""
The user replied: "{msg}" when asked for their total budget.

They must enter a number like 30000.
Politely ask them to enter their budget as a number only, and give an example.
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

        state["budget"] = int(nums[-1])
        recs = filter_and_rank(state["interest"], state["delivery"], state["budget"])

        if not recs:
            prompt = f"""
The user entered a budget of {state['budget']} for subject "{state['interest']}" with delivery "{state['delivery']}",
but no universities in the dataset fit within this budget.

Politely explain that nothing was found and ask them to enter a higher budget so you can try again.
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

        state["step"] = 4
        return call_llm(SYSTEM_PROMPT, llm_prompt(3, state, msg=msg, recs=recs)), state

    if step == 4:
        lower = msg.lower()
        if any(word in lower for word in ["no", "not now", "i am good", "that is enough"]):
            state["step"] = 5
            prompt = f"""
The user replied: "{msg}" which indicates they are done exploring options.

Thank them briefly and ask:
"How would you rate this experience from 1 to 5?"
"""
            return call_llm(SYSTEM_PROMPT, prompt), state
        else:
            state = {"step": 1, "interest": None, "interest_raw": None, "delivery": None, "budget": None}
            prompt = f"""
The user replied: "{msg}" which suggests they want to keep exploring.

Respond warmly and ask:
"Sure, what subject would you like to explore next?"
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

    if step == 5:
        digits = re.findall(r"[1-5]", msg)
        if digits:
            rating_val = int(digits[-1])
            state["rating"] = rating_val
            state["step"] = 6
            return call_llm(SYSTEM_PROMPT, llm_prompt(5, state, msg=str(rating_val))), state

        prompt = f"""
The user replied: "{msg}" when asked to rate the experience from 1 to 5.

Politely tell them that the rating must include a number between 1 and 5 and ask them again to rate from 1 to 5.
"""
        return call_llm(SYSTEM_PROMPT, prompt), state

    if step == 6:
        lower = msg.lower()
        if any(word in lower for word in ["yes", "sure", "ok", "okay", "yep", "yeah"]):
            state = {"step": 1, "interest": None, "interest_raw": None, "delivery": None, "budget": None}
            prompt = f"""
The user replied: "{msg}" which means they want to explore more.

Respond cheerfully and ask:
"Great, what subject would you like to explore next?"
"""
            return call_llm(SYSTEM_PROMPT, prompt), state
        if any(word in lower for word in ["no", "not now", "i am done", "that's all", "that is all"]):
            prompt = f"""
The user replied: "{msg}" which means they are finished.

Thank them and end the conversation politely.
"""
            return call_llm(SYSTEM_PROMPT, prompt), state

        prompt = f"""
The user replied: "{msg}" when asked if they would like to explore another subject.

It is not clearly yes or no.
Politely ask them to reply with yes or no.
"""
        return call_llm(SYSTEM_PROMPT, prompt), state


# ----------------------------------------------------
# 7. GRADIO UI WITH CUSTOM STYLING
# ----------------------------------------------------
CUSTOM_CSS = """
/* Header row styling */
#uniguide-header {
    background-color: #cfe8d6;
    text-align: center;
    padding: 10px 12px;
    border-radius: 8px;
    font-weight: 600;
}

/* Chatbot area: taller and scrollable, no max-height cap */
#chatbot {
    height: 650px;           /* increase visible area */
    overflow-y: auto;        /* scroll inside if content is longer */
}

/* User and bot message bubbles */
#chatbot .message.user {
    background-color: #e0f7f7;
    color: #000000;
    box-shadow: 0 2px 6px rgba(0, 0, 0, 0.18);
}

#chatbot .message.bot {
    background-color: #ffffff;
    color: #000000;
    box-shadow: 0 2px 6px rgba(0, 0, 0, 0.18);
}

#chatbot .message {
    border-radius: 10px;
    margin-bottom: 6px;
}
"""

with gr.Blocks(css=CUSTOM_CSS) as app:
    gr.Markdown("##  UniGuide", elem_id="uniguide-header")

    status_text = (
        "✅ University data loaded from Excel file."
        if DATA_LOADED
        else "❗ Could not load university data. Please verify the Excel path."
    )
    gr.Markdown(status_text)

    chatbot = gr.Chatbot(elem_id="chatbot", label="UniGuide Chat")
    user_box = gr.Textbox(label="Your message", placeholder="Type here and press Enter...")

    state = gr.State({"step": 0, "interest": None, "interest_raw": None, "delivery": None, "budget": None})

    def init_chat(state):
        reply, s = step_logic("", state)
        return [[None, reply]], s

    app.load(init_chat, [state], [chatbot, state])

    def chat(msg, chat_history, state):
        reply, new_state = step_logic(msg, state)
        chat_history.append([msg, reply])
        return "", chat_history, new_state

    user_box.submit(chat, [user_box, chatbot, state], [user_box, chatbot, state])

app.launch(share=True)